# Notebook 04 — Reliability-Aware Agentic RAG System

## Overview

This notebook demonstrates the reliability extension of the multi-agent RAG baseline developed in Notebook 02. The baseline system retrieves and synthesises answers without evaluating the quality of the underlying evidence, producing hallucinations and overconfident responses in the failure cases identified in the failure analysis.

The extended system introduces a **two-gate reliability architecture** that intercepts the pipeline at two checkpoints: before answer generation (pre-synthesis gate) and after (post-synthesis gate). Between the two gates, an adaptive recovery mechanism modifies retrieval behaviour when evidence quality is insufficient.

## Scope

This notebook covers the following mechanisms:

| Mechanism | Component | Role |
|---|---|---|
| **A** | EvidenceAnalyst | Pre-synthesis gate: estimates whether retrieved evidence is sufficient to proceed |
| **G** | RecoveryAgent | Adaptive recovery: rewrites the query or switches retrieval strategy on failure |
| **E** | AbstentionGate | Unified terminal handler: issues a structured abstention for all failure paths |
| **B** | GroundednessVerifier | Post-synthesis verification: checks whether the answer is grounded in evidence |
| **H** | TrustScorer | Post-synthesis gate: aggregates signals and routes to abstention if trust is insufficient |

> Mechanisms **B** and **H** are integrated as injectable components. Their placeholder interfaces are defined in this notebook; the implementations are provided separately.

## System Architecture

```
Query
  └─► QueryProfiler
           └─► RetrievalCoordinator  (BM25 · Dense · GraphRAG · Weighted RRF)
                    └─► [A] EvidenceAnalyst
                              ├── sufficient ─────────────────────────► AnswerSynthesizer
                              │                                                │
                              │                                          [B] GroundednessVerifier
                              │                                                │
                              │                                          [H] TrustScorer
                              │                                                ├── trust OK ──► Final Response
                              │                                                └── trust low ──┐
                              │                                                                 │
                              ├── insufficient ──► [G] RecoveryAgent                           │
                              │                         ├── retry ──► RetrievalCoordinator     │
                              │                         └── exhausted ──────────────────────┐  │
                              │                                                              │  │
                              └── hard floor ───────────────────────────────────────────────▼──▼
                                                                                     [E] AbstentionGate
```

## Prerequisites

- Notebook 02 completed: baseline retrievers (BM25, Dense, GraphRAG) built and stored in Google Drive
- Shared Drive folder `Adv_GenAI_FS26` accessible and paths configured in Section 2

## Section 1 — Installation

In [ ]:
!pip install --quiet --upgrade pip setuptools wheel
!pip install --quiet --upgrade --no-cache-dir \
  "git+https://github.com/dhub100/advanced-genai-rag.git@evidence_sufficiency#subdirectory=baseline/package/"

# Verify the reliability module is present
import importlib, sys
if "rag" in sys.modules:
    del sys.modules["rag"]
import rag.reliability
print("✓ rag.reliability module found.")

In [ ]:
import json
import time
import pathlib
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Section 2 — Configuration

Edit the `ROOT` path if your Drive layout differs from the default.

| Variable | Description |
|---|---|
| `PATH_BM25_PICKLE` | Serialised `QEBM25` object built in Notebook 02 |
| `PATH_DENSE_LOADER` | `load_dense_fixed.py` generated during corpus indexing |
| `PATH_GRAG_LOADER` | `load_graphrag.py` generated during graph construction |
| `PATH_QA` | Bilingual benchmark JSON (25 EN + 25 DE question-answer pairs) |
| `PATH_QRELS` | Folder of per-document relevance score JSON files |

In [ ]:
ROOT = pathlib.Path("/content/drive/MyDrive/Adv_GenAI_FS26")

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA           = ROOT / "advanced-genai-rag/baseline/benchmark/benchmark_qa_bilingual.json"
PATH_QRELS        = ROOT / "advanced-genai-rag/baseline/benchmark/score/fixed_size"

for name, p in [("BM25", PATH_BM25_PICKLE), ("Dense loader", PATH_DENSE_LOADER),
                ("GraphRAG loader", PATH_GRAG_LOADER), ("QA benchmark", PATH_QA)]:
    status = "✓" if pathlib.Path(p).exists() else "✗ NOT FOUND"
    print(f"  {status}  {name}: {p}")

## Section 3 — Load Baseline System

Loads the pre-built retrieval components from Google Drive. Refer to Notebook 02 for detailed explanations of each component.

Two compatibility patches are applied before loading:
1. **`torch.is_available` patch** — the generated `load_dense_fixed.py` calls `torch.is_available()` instead of `torch.cuda.is_available()`. We register the alias before the file is imported.
2. **`__main__` pickle patch** — the BM25 object was pickled from a notebook where `BilingualBM25` and `QEBM25` were defined in `__main__`. We re-register the package classes under `__main__` so that `pickle.load` can resolve them.

In [ ]:
import __main__
import torch

# Patch 1: torch.is_available() alias
if not hasattr(torch, 'is_available'):
    torch.is_available = torch.cuda.is_available

# Patch 2: re-register pickled classes in __main__
from rag.retrieval.agents.bm25 import BilingualBM25, QEBM25
from rag.retrieval.translator import EnDeTranslator
__main__.BilingualBM25 = BilingualBM25
__main__.QEBM25 = QEBM25
__main__.EnDeTranslator = EnDeTranslator

from rag.retrieval.agents.bm25 import load_bm25_fixed_qe
from rag.retrieval.agents.dense import DenseRetriever, load_dense_fixed
from rag.retrieval.agents.graph import GraphAgent, load_graph_rag
from rag.retrieval.agents.answer_synthesizer import AnswerSynthesizerAgent
from rag.retrieval.orchestrator import Orchestrator

print("Loading BM25...")
bm25_fixed_qe = load_bm25_fixed_qe(bm25_pickle_path=str(PATH_BM25_PICKLE))
print("Loading Dense retriever...")
dense_fixed   = load_dense_fixed(dense_loader_path=str(PATH_DENSE_LOADER))
print("Loading GraphRAG...")
graph_rag     = load_graph_rag(loader_path=str(PATH_GRAG_LOADER))
print("Loading AnswerSynthesizer (Mistral-7B)...")
synthesizer   = AnswerSynthesizerAgent()

baseline_orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed,
    graph=graph_rag, synthesizer=synthesizer
)
print("✓ Baseline system ready.")

## Section 4 — Evidence Sufficiency Estimation [Mechanism A]

### Problem

The baseline passes retrieved documents directly to Mistral-7B without assessing whether they contain sufficient relevant information. This is the root cause of two failure types identified in the failure analysis: hallucinations arise when weak evidence is treated as valid input, and overconfidence failures arise when the system returns a confident answer based on tangentially related material.

### Solution: EvidenceSufficiencyChecker

The `EvidenceSufficiencyChecker` evaluates three independent signals on the retrieved document set **before** any answer is generated:

| Signal | Method | Threshold |
|---|---|---|
| `semantic_coverage` | Average cosine similarity between the query embedding and top-k document embeddings | ≥ 0.35 |
| `chunk_support_count` | Number of documents whose individual cosine similarity to the query exceeds 0.40 | ≥ 3 |
| `aspect_coverage_ratio` | Fraction of query key-terms (stopword-filtered) present in the combined retrieved text | ≥ 0.50 |
| `sufficiency_score` | Weighted composite of the three signals above (weights: 0.40 / 0.35 / 0.25) | ≥ 0.45 |

A **hard floor** of 0.20 bypasses the recovery loop and triggers immediate abstention — evidence below this threshold is too weak to justify additional retrieval attempts.

Three independent signals are used rather than a single threshold because each captures a distinct failure mode: `semantic_coverage` measures overall topical proximity, `chunk_support_count` ensures multiple independent documents corroborate the query, and `aspect_coverage_ratio` detects when specific query terms are absent from the evidence — each maps to a different recovery strategy.

> The `EvidenceSufficiencyChecker` reuses the `multilingual-E5-large` embeddings already loaded by the Dense retriever. No additional model is loaded.

In [ ]:
from rag.reliability.evidence_sufficiency import EvidenceSufficiencyChecker, EvidenceReport

checker = EvidenceSufficiencyChecker(embed_fn=dense_fixed.embed)
print("✓ EvidenceSufficiencyChecker ready.")

In [ ]:
def show_evidence_report(query: str, strategy: str = "confidence", top_k: int = 5):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=top_k)
    report = checker.check(query, result["documents"])

    print(f"Query  : {query}")
    print(f"{'─'*62}")
    print(f"  semantic_coverage     : {report.semantic_coverage:.3f}   (threshold ≥ 0.35)")
    print(f"  chunk_support_count   : {report.chunk_support_count}       (threshold ≥ 3)")
    print(f"  aspect_coverage_ratio : {report.aspect_coverage_ratio:.3f}   (threshold ≥ 0.50)")
    print(f"{'─'*62}")
    print(f"  sufficiency_score     : {report.score:.3f}   → sufficient = {report.sufficient}")
    print(f"  failure_type          : {report.failure_type}")
    if report.missing_aspects:
        print(f"  missing_aspects       : {report.missing_aspects}")
    return report

In [ ]:
print("=" * 62)
print("EXAMPLE 1 — likely sufficient evidence")
print("=" * 62)
_ = show_evidence_report("Who at ETH received ERC grants?")

print()
print("=" * 62)
print("EXAMPLE 2 — known baseline failure (hallucination case)")
print("=" * 62)
_ = show_evidence_report("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("EXAMPLE 3 — ambiguous query")
print("=" * 62)
_ = show_evidence_report("What research is ETH famous for?")

## Section 5 — Recovery Agent [Mechanism G — Adaptive Requirement]

### Design rationale

A generic retry with an unchanged query would explore no new evidence. The `RecoveryAgent` diagnoses which signal was the dominant failure contributor and selects a targeted corrective action that modifies the system's subsequent retrieval behaviour. This satisfies the adaptation requirement: the orchestration logic changes dynamically based on observed evidence quality rather than following a fixed script.

The recovery policy maps each failure type to a distinct corrective action:

| `failure_type` | Recovery action | Rationale |
|---|---|---|
| `coverage_low` | LLM-based query rewrite | Low aspect coverage indicates the query is too narrow; rephrasing with synonyms or broader terms improves recall |
| `few_docs` | Strategy switch (Confidence → Voting → Waterfall) | A different fusion strategy surfaces complementary results from underutilised agents |
| `low_similarity` | PRF expansion via QEBM25 | Low semantic similarity indicates a vocabulary mismatch; pseudo-relevance feedback expands the query using top-document terms |
| `exhausted` | Signal AbstentionGate | Further attempts are unlikely to yield better evidence given the retrieval corpus |

A maximum of **two recovery attempts** is enforced to bound worst-case latency.

### Query rewriting

LLM-based query rewriting uses the OpenAI API and requires a key stored as a Colab Secret:
1. Click the **🔑 Secrets** icon in the left Colab sidebar
2. Add a secret named `OPENAI_API_KEY` with your key as the value
3. Toggle **Notebook access** on

The key is stored in your personal Colab account and is never written to the notebook file. Without a key, a rule-based fallback appends the missing key-terms to the original query.

In [ ]:
from rag.reliability.recovery import RecoveryAgent

openai_client = None
try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if openai_api_key:
        from openai import OpenAI
        openai_client = OpenAI(api_key=openai_api_key)
        print("✓ OpenAI client ready — LLM-based query rewriting enabled.")
    else:
        print("ℹ  OPENAI_API_KEY secret not set — rule-based query rewrite fallback will be used.")
except Exception:
    print("ℹ  Could not read Colab secrets — rule-based query rewrite fallback will be used.")

recovery_agent = RecoveryAgent(max_retries=2, openai_client=openai_client)
print("✓ RecoveryAgent ready.")

In [ ]:
def show_recovery_actions(query: str, strategy: str = "confidence"):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=5)
    report = checker.check(query, result["documents"])

    print(f"Query        : {query}")
    print(f"sufficient   : {report.sufficient}  |  score={report.score:.3f}  |  failure='{report.failure_type}'")

    if report.sufficient:
        print("→ Evidence is sufficient — no recovery needed.")
        return

    current_strategy = strategy
    current_query    = query
    for attempt in range(recovery_agent.max_retries + 1):
        action = recovery_agent.choose_action(report, attempt=attempt, current_strategy=current_strategy)
        print(f"  {action.trace_entry}")
        if action.type == "abstain":
            break
        if action.type == "rewrite_query":
            current_query = recovery_agent.rewrite_query(current_query, report.missing_aspects)
            print(f"    → Rewritten query: '{current_query}'")
        elif action.type == "switch_strategy":
            current_strategy = action.new_strategy


print("=" * 62)
print("RECOVERY — few_docs failure")
print("=" * 62)
show_recovery_actions("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("RECOVERY — coverage_low failure")
print("=" * 62)
show_recovery_actions("What are ETH sustainability initiatives for 2030?")

## Section 6 — Abstention Gate [Mechanism E]

### Design rationale

Returning an empty string or a generic "I don't know" provides no actionable information to the user or to the evaluation pipeline. The `AbstentionGate` produces a structured `AbstentionResponse` containing a machine-readable `trigger`, a human-readable `reason`, the list of `missing_aspects` that were absent from the evidence, a `confidence` of 0.0, and the full decision `trace` accumulated up to that point.

This structure enables fine-grained evaluation metrics — in particular the distinction between **correct abstentions** (the answer was genuinely unavailable) and **false abstentions** (the answer existed but thresholds were miscalibrated).

### Two entry paths

The AbstentionGate is the **unified terminal failure state**, reached from two independent paths:

| Path | Trigger | Method |
|---|---|---|
| Retrieval failure | A hard floor or G exhausted | `abstain_evidence(query, report, trace)` |
| Synthesis failure | H trust score below threshold | `abstain_low_trust(query, trust_score, groundedness_score, trace)` |

In [ ]:
from rag.reliability.abstention import AbstentionMechanism, AbstentionResponse

abstainer = AbstentionMechanism()
print("✓ AbstentionMechanism ready.")

In [ ]:
def show_abstention_evidence(query: str):
    result = baseline_orchestrator.run(strategy="confidence", query=query, top_k=5)
    report = checker.check(query, result["documents"])
    trace  = [
        f"EvidenceCheck: score={report.score:.3f}, failure='{report.failure_type}'",
        "Recovery attempts exhausted."
    ]
    abstention = abstainer.abstain_evidence(query, report, trace)

    print(f"Query      : {query}")
    print(f"Trigger    : {abstention.trigger}")
    print(f"Reason     : {abstention.reason}")
    print(f"Missing    : {abstention.missing_aspects}")
    print(f"Confidence : {abstention.confidence}")
    print("Trace:")
    for e in abstention.trace:
        print(f"  {e}")


print("=" * 62)
print("ABSTENTION — evidence_failure path (A → G → E)")
print("=" * 62)
show_abstention_evidence("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("ABSTENTION — trust_failure path (B → H → E)")
print("=" * 62)
# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Replace the placeholder values with the real trust_score and
# groundedness_score produced by H and B respectively.
# ─────────────────────────────────────────────────────────────
abstention_trust = abstainer.abstain_low_trust(
    query="Who was president of ETH in 2003?",
    trust_score=0.0,          # placeholder — replace with H output
    groundedness_score=0.0,   # placeholder — replace with B output
    trace=["[H placeholder] trust_score below threshold"]
)
print(f"Trigger    : {abstention_trust.trigger}")
print(f"Reason     : {abstention_trust.reason}")
print(f"Trust      : {abstention_trust.trust_score}")

## Section 7 — Reliable Orchestrator (A + G + E)

The `ReliableOrchestrator` is the policy controller that coordinates all reliability mechanisms. It wraps the existing `Orchestrator` and inserts the evidence-based loop between retrieval and synthesis, implementing the decision policy described in the system architecture.

Mechanisms B and H are injected as optional parameters. When set to `None`, the post-synthesis gate is bypassed gracefully and the answer is returned directly after synthesis — allowing the pre-synthesis mechanisms (A, G, E) to be evaluated independently before B and H are available.

### Decision policy summary

| State | Condition | Next action |
|---|---|---|
| After retrieval | `sufficiency_score ≥ 0.45` | Proceed to synthesis |
| After retrieval | `sufficiency_score < 0.20` (hard floor) | Abstain immediately |
| After retrieval | `0.20 ≤ score < 0.45`, attempts < 2 | Recover → retry retrieval |
| After retrieval | `0.20 ≤ score < 0.45`, attempts ≥ 2 | Abstain |
| After synthesis | `trust_score ≥ 0.45` | Return final response |
| After synthesis | `trust_score < 0.45` | Abstain (trust_failure) |

In [ ]:
from rag.reliability.reliable_orchestrator import ReliableOrchestrator, ReliableResponse

# ─────────────────────────────────────────────────────────────
# Introduce code here for B (GroundednessVerifier)
# Replace None with an instance of B once implemented.
# ─────────────────────────────────────────────────────────────
groundedness_verifier = None

# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Replace None with an instance of H once implemented.
# ─────────────────────────────────────────────────────────────
trust_scorer = None

reliable_orchestrator = ReliableOrchestrator(
    orchestrator=baseline_orchestrator,
    embed_fn=dense_fixed.embed,
    max_retries=2,
    openai_client=openai_client,
    groundedness_verifier=groundedness_verifier,
    trust_scorer=trust_scorer,
)
print("✓ ReliableOrchestrator ready.")

In [ ]:
def run_and_show(query: str, strategy: str = "confidence"):
    t0       = time.time()
    response = reliable_orchestrator.run(query=query, strategy=strategy, top_k=5)
    elapsed  = time.time() - t0

    print(f"Query      : {query}")
    print(f"Abstained  : {response.abstained}  |  Recoveries: {response.recovery_attempts}  |  {elapsed:.2f}s")

    if response.abstained:
        print(f"Trigger    : {response.abstention.trigger}")
        print(f"Reason     : {response.abstention.reason}")
    else:
        print(f"Answer     : {response.answer}")
        if response.evidence_report:
            print(f"Evidence   : score={response.evidence_report.score:.3f}")

    print("Trace (last 5 entries):")
    for entry in response.trace[-5:]:
        print(f"  {entry}")
    print()

In [ ]:
test_queries = [
    ("Who was president of ETH in 2003?",              "hallucination case — baseline failure"),
    ("Who at ETH received ERC grants?",                "likely answerable"),
    ("Who were the rectors of ETH between 2017–2022?", "ranking failure — baseline failure"),
    ("Wer war 2003 Präsident der ETH?",                "cross-lingual (German)"),
]

for query, label in test_queries:
    print("=" * 62)
    print(f"{label}")
    print("=" * 62)
    run_and_show(query)

## Section 8 — Benchmark Evaluation

Runs the reliable orchestrator over all 50 benchmark queries (25 EN + 25 DE) and reports abstention rate, recovery statistics, and per-query outcomes.

In [ ]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions, gold_answers = [], []
for x in qa_data:
    questions.append(x["question"])
    gold_answers.append(x["answer"])
    questions.append(x["question_de"])
    gold_answers.append(x["answer_de"])

print(f"Loaded {len(questions)} queries ({len(qa_data)} bilingual pairs).")

In [ ]:
reliable_results = []

for q, a in tqdm(zip(questions, gold_answers), total=len(questions), desc="Reliable pipeline"):
    t0 = time.time()
    r  = reliable_orchestrator.run(query=q, strategy="confidence", top_k=5)
    reliable_results.append({
        "query":      q,
        "gold":       a,
        "answer":     r.answer,
        "abstained":  r.abstained,
        "trigger":    r.abstention.trigger if r.abstained else None,
        "recoveries": r.recovery_attempts,
        "ev_score":   r.evidence_report.score if r.evidence_report else None,
        "latency":    time.time() - t0,
    })

rel_df = pd.DataFrame(reliable_results)
print("Done.")

In [ ]:
n            = len(reliable_results)
abstained_df = rel_df[rel_df["abstained"]]
recovered_df = rel_df[rel_df["recoveries"] > 0]
answered_df  = rel_df[~rel_df["abstained"]]

print(f"{'─'*40}")
print(f"  Total queries    : {n}")
print(f"  Answered         : {len(answered_df):3d}  ({len(answered_df)/n:.1%})")
print(f"  Abstained        : {len(abstained_df):3d}  ({len(abstained_df)/n:.1%})")
print(f"  Had recovery     : {len(recovered_df):3d}  ({len(recovered_df)/n:.1%})")
print(f"  Avg latency      : {rel_df['latency'].mean():.2f}s")
print(f"  Avg ev_score     : {rel_df['ev_score'].dropna().mean():.3f}")
print(f"{'─'*40}")

if len(abstained_df) > 0:
    print("\nAbstention trigger breakdown:")
    print(abstained_df["trigger"].value_counts().to_string())
    print("\nAbstained queries:")
    for _, row in abstained_df.iterrows():
        ev = f"{row['ev_score']:.2f}" if row['ev_score'] is not None else "N/A"
        print(f"  [{row['trigger']:20s}] ev={ev}  {row['query'][:60]}")

In [ ]:
if len(recovered_df) > 0:
    answered_after  = recovered_df[~recovered_df["abstained"]]
    abstained_after = recovered_df[recovered_df["abstained"]]

    print("Recovery outcomes:")
    print(f"  Answered after recovery  : {len(answered_after)}")
    print(f"  Abstained after recovery : {len(abstained_after)}")

    if len(answered_after) > 0:
        print("\nQueries where recovery produced an answer:")
        for _, row in answered_after.iterrows():
            print(f"  [recoveries={int(row['recoveries'])}] ev={row['ev_score']:.2f}  {row['query'][:60]}")
else:
    print("No recovery attempts triggered on this benchmark run.")